# Modellvergleich am Ende von Tag 28

**Vorhersagefrage:** Welche am Ende von Tag 28 noch angemeldeten Personen
brechen den Kurs später ab oder bestehen ihn nicht?

**Anwendungsfrage:** Welche Personen sollten innerhalb einer
Kurspräsentation bei begrenzter Beratungskapazität zuerst ein
Unterstützungsangebot erhalten?

Wir vergleichen Kurs- und Anmeldedaten, VLE-Aktivität und frühe
Assessments. Average Precision (AP) bewertet die gesamte Rangfolge.
Für die praktische Priorisierung betrachten wir außerdem Trefferquote
und Recall bei einem Hinweisbudget von 10 % beziehungsweise 20 %.

Zu Beginn der Analyse untersuchten wir auch größere Hinweisanteile
von 60 % und 70 %, weil ein hoher Recall im Vordergrund stand.
Der Schwerpunkt auf wenige, möglichst zutreffende Hinweise entstand
erst im Verlauf der Auswertung. Die entsprechenden früheren
Berechnungen bleiben im Notebook nachvollziehbar.

Training und Bewertung sind nach Personen getrennt: Kurseinträge
derselben Person dürfen nicht auf beiden Seiten einer Aufteilung liegen.
Die kursweise Vergabe der Hinweise wurde bislang explorativ in der
Entwicklungsgruppe untersucht; der Prüfteil bewertet die
kursübergreifende Rangfolge.

In [2]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Funktioniert, wenn das Notebook vom Projektordner oder vom Ordner
# "notebooks" aus gestartet wird.
projektordner = Path.cwd()
if projektordner.name.lower() == "notebooks":
    projektordner = projektordner.parent

datei = projektordner / "data" / "processed" / "oulad_tag28_modellbasis.csv"
assert datei.exists(), f"Datei nicht gefunden: {datei}"

basis = pd.read_csv(datei)
schluessel = ["code_module", "code_presentation", "id_student"]

assert not basis.duplicated(schluessel).any()
assert not basis.isna().any().any()

# Rund 20 % der Personen bleiben für eine spätere, einmalige Prüfung zurück.
aufteilung = GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=2026
)
entwicklungs_idx, pruef_idx = next(
    aufteilung.split(basis, groups=basis["id_student"])
)

entwicklung = basis.iloc[entwicklungs_idx].reset_index(drop=True)
pruefteil = basis.iloc[pruef_idx].reset_index(drop=True)

assert set(entwicklung["id_student"]).isdisjoint(pruefteil["id_student"])

print("Entwicklung:", len(entwicklung))
print("Zurückgelegter Prüfteil:", len(pruefteil))
print(
    "Anteil Abbruch oder Fail in der Entwicklung:",
    round(entwicklung["abbruch_oder_fail"].mean(), 3)
)

Entwicklung: 21948
Zurückgelegter Prüfteil: 5474
Anteil Abbruch oder Fail in der Entwicklung: 0.438


In [3]:
ziel = entwicklung["abbruch_oder_fail"].astype(int)
hinweis = entwicklung["assessments_fehlend_bis28"] > 0

anzahl_hinweise = int(hinweis.sum())
treffer = int(ziel[hinweis].sum())
alle_betroffenen = int(ziel.sum())

print("Hinweise:", anzahl_hinweise)
print("Davon Abbruch oder Fail:", treffer)
print("Hinweisanteil:", round(anzahl_hinweise / len(entwicklung), 3))
print("Trefferquote:", round(treffer / anzahl_hinweise, 3))
print("Recall:", round(treffer / alle_betroffenen, 3))

Hinweise: 2065
Davon Abbruch oder Fail: 1817
Hinweisanteil: 0.094
Trefferquote: 0.88
Recall: 0.189


In [4]:
from sklearn.model_selection import StratifiedGroupKFold

# Die Folds versuchen ähnliche Zielanteile zu erreichen und halten
# Kurseinträge derselben Person zusammen.
cv = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=42
)
falten = list(
    cv.split(entwicklung, ziel, groups=entwicklung["id_student"])
)

for nummer, (train_idx, valid_idx) in enumerate(falten, start=1):
    personen_train = set(entwicklung.iloc[train_idx]["id_student"])
    personen_valid = set(entwicklung.iloc[valid_idx]["id_student"])
    assert personen_train.isdisjoint(personen_valid)

    print(
        f"Fold {nummer}: {len(valid_idx)} Kurseinträge, "
        f"Zielanteil {ziel.iloc[valid_idx].mean():.3f}"
    )

Fold 1: 4389 Kurseinträge, Zielanteil 0.438
Fold 2: 4390 Kurseinträge, Zielanteil 0.438
Fold 3: 4390 Kurseinträge, Zielanteil 0.438
Fold 4: 4389 Kurseinträge, Zielanteil 0.438
Fold 5: 4390 Kurseinträge, Zielanteil 0.438


In [5]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

kategorien = ["code_module", "code_presentation"]
grundmerkmale = kategorien + ["anmeldetag"]

vle_merkmale = [
    "klicks_tag_0_bis_6",
    "klicks_tag_7_bis_14",
    "klicks_tag_15_bis_28",
]

assessment_merkmale = [
    "assessments_faellig_bis28",
    "abgaben_bis28",
    "assessments_fehlend_bis28",
    "banked_faellige_bis28",
]

varianten = {
    "Kurs + Anmeldung": grundmerkmale,
    "Kurs + VLE": grundmerkmale + vle_merkmale,
    "Kurs + Assessments": grundmerkmale + assessment_merkmale,
    "Alle Merkmale": grundmerkmale + vle_merkmale + assessment_merkmale,
}

ergebnisse = []

def bewerte_modell(name, merkmale, klassifikator):
    """Trainiert ein Modell auf denselben fünf Folds und misst seine Rangfolge."""
    zahlen = [spalte for spalte in merkmale if spalte not in kategorien]

    for fold_nummer, (train_idx, valid_idx) in enumerate(falten, start=1):
        vorbereitung = ColumnTransformer(
            transformers=[
                (
                    "kategorien",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                    kategorien,
                ),
                ("zahlen", StandardScaler(), zahlen),
            ]
        )

        modell = Pipeline(
            steps=[
                ("vorbereitung", vorbereitung),
                ("klassifikator", klassifikator),
            ]
        )

        modell.fit(
            entwicklung.iloc[train_idx][merkmale],
            ziel.iloc[train_idx],
        )

        risiko = modell.predict_proba(
            entwicklung.iloc[valid_idx][merkmale]
        )[:, 1]
        wahr = ziel.iloc[valid_idx].to_numpy()

        zeile = {
            "Verfahren": name,
            "Fold": fold_nummer,
            "AP": average_precision_score(wahr, risiko),
        }

        # Die höchsten Risikowerte erhalten zuerst einen Hinweis.
        sortierung = np.argsort(-risiko, kind="stable")
        for anteil in (60, 70):
            anzahl = int(np.ceil(len(wahr) * anteil / 100))
            treffer = int(wahr[sortierung[:anzahl]].sum())

            zeile[f"Recall_{anteil}"] = treffer / wahr.sum()
            zeile[f"Trefferquote_{anteil}"] = treffer / anzahl

        ergebnisse.append(zeile)

    print("Fertig:", name)


# Vier Varianten derselben logistischen Regression zeigen den Beitrag
# der jeweiligen Datengruppe.
for name, merkmale in varianten.items():
    bewerte_modell(
        f"LogReg: {name}",
        merkmale,
        LogisticRegression(max_iter=1000),
    )

# Ein flexibleres Verfahren prüft, ob Kombinationen der Merkmale helfen.
bewerte_modell(
    "Boosting: Alle Merkmale",
    varianten["Alle Merkmale"],
    HistGradientBoostingClassifier(random_state=42),
)

vergleich = pd.DataFrame(ergebnisse)

uebersicht = (
    vergleich.groupby("Verfahren")
    .agg(
        AP=("AP", "mean"),
        AP_Streuung=("AP", "std"),
        Recall_60=("Recall_60", "mean"),
        Trefferquote_60=("Trefferquote_60", "mean"),
        Recall_70=("Recall_70", "mean"),
        Trefferquote_70=("Trefferquote_70", "mean"),
    )
    .sort_values("AP", ascending=False)
    .round(3)
)

display(uebersicht)

Fertig: LogReg: Kurs + Anmeldung
Fertig: LogReg: Kurs + VLE
Fertig: LogReg: Kurs + Assessments
Fertig: LogReg: Alle Merkmale
Fertig: Boosting: Alle Merkmale


,AP,AP_Streuung,Recall_60,Trefferquote_60,Recall_70,Trefferquote_70
Verfahren,,,,,,
Boosting: Alle Merkmale,0.701,0.009,0.765,0.558,0.837,0.523
LogReg: Alle Merkmale,0.684,0.004,0.749,0.546,0.831,0.520
LogReg: Kurs + Assessments,0.641,0.009,0.703,0.513,0.791,0.495
LogReg: Kurs + VLE,0.632,0.006,0.746,0.544,0.826,0.517
LogReg: Kurs + Anmeldung,0.499,0.009,0.664,0.485,0.758,0.474


In [6]:
# Im ersten Vergleich wurden nur Kennzahlen gespeichert.
# Hier berechnen wir die Vorhersagen des besten Modells noch einmal:
# Jede Person erhält einen Risikowert aus einem Fold, in dem sie nicht
# zum Training gehörte.
alle_spalten = varianten["Alle Merkmale"]
zahlenspalten = [
    spalte for spalte in alle_spalten
    if spalte not in kategorien
]

risiken_cv = np.full(len(entwicklung), np.nan)

for fold_nummer, (train_idx, valid_idx) in enumerate(falten, start=1):
    vorbereitung = ColumnTransformer(
        transformers=[
            (
                "kategorien",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                kategorien,
            ),
            ("zahlen", StandardScaler(), zahlenspalten),
        ]
    )

    modell = Pipeline(
        steps=[
            ("vorbereitung", vorbereitung),
            ("klassifikator", HistGradientBoostingClassifier(random_state=42)),
        ]
    )

    modell.fit(
        entwicklung.iloc[train_idx][alle_spalten],
        ziel.iloc[train_idx],
    )

    risiken_cv[valid_idx] = modell.predict_proba(
        entwicklung.iloc[valid_idx][alle_spalten]
    )[:, 1]

    print(f"Fold {fold_nummer} fertig")

assert not np.isnan(risiken_cv).any()

Fold 1 fertig
Fold 2 fertig
Fold 3 fertig
Fold 4 fertig
Fold 5 fertig


In [7]:
details = []

for fold_nummer, (_, valid_idx) in enumerate(falten, start=1):
    wahr = ziel.iloc[valid_idx].to_numpy()
    risiko = risiken_cv[valid_idx]

    # Innerhalb jedes Validierungs-Folds nach Risiko sortieren.
    reihenfolge = np.argsort(-risiko, kind="stable")

    for anteil in (10, 20, 30, 40, 50, 60, 70, 80):
        anzahl_hinweise = int(np.ceil(len(wahr) * anteil / 100))
        treffer = int(wahr[reihenfolge[:anzahl_hinweise]].sum())

        details.append({
            "Fold": fold_nummer,
            "Hinweisanteil": anteil,
            "Recall": treffer / wahr.sum(),
            "Trefferquote": treffer / anzahl_hinweise,
        })

detailtabelle = pd.DataFrame(details)

uebersicht_anteile = (
    detailtabelle.groupby("Hinweisanteil")
    .agg(
        Recall=("Recall", "mean"),
        Recall_Streuung=("Recall", "std"),
        Trefferquote=("Trefferquote", "mean"),
    )
    .round(3)
)

display(uebersicht_anteile)

,Recall,Recall_Streuung,Trefferquote
Hinweisanteil,,,
10,0.202,0.003,0.883
20,0.349,0.007,0.764
30,0.473,0.006,0.690
40,0.580,0.004,0.634
50,0.678,0.006,0.593
60,0.765,0.005,0.558
70,0.837,0.007,0.523
80,0.902,0.004,0.494


In [8]:
def erstelle_modell(merkmale, klassifikator):
    """Erstellt dieselbe Vorverarbeitung wie im Modellvergleich."""
    zahlenspalten = [
        spalte for spalte in merkmale
        if spalte not in kategorien
    ]

    vorbereitung = ColumnTransformer(
        transformers=[
            (
                "kategorien",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                kategorien,
            ),
            ("zahlen", StandardScaler(), zahlenspalten),
        ]
    )

    return Pipeline(
        steps=[
            ("vorbereitung", vorbereitung),
            ("klassifikator", klassifikator),
        ]
    )


# Das Boosting ist unser ausgewähltes Modell. Kurs + Anmeldung
# bleibt als einfache Referenz für die abschließende Prüfung.
kandidaten = {
    "Referenz: Kurs + Anmeldung": (
        varianten["Kurs + Anmeldung"],
        LogisticRegression(max_iter=1000),
    ),
    "Boosting: Alle Merkmale": (
        varianten["Alle Merkmale"],
        HistGradientBoostingClassifier(random_state=42),
    ),
}

risiken_test = {}

for name, (merkmale, klassifikator) in kandidaten.items():
    modell = erstelle_modell(merkmale, klassifikator)

    modell.fit(
        entwicklung[merkmale],
        ziel,
    )

    risiken_test[name] = modell.predict_proba(
        pruefteil[merkmale]
    )[:, 1]

    print("Trainiert und vorhergesagt:", name)

Trainiert und vorhergesagt: Referenz: Kurs + Anmeldung
Trainiert und vorhergesagt: Boosting: Alle Merkmale


In [9]:
ziel_test = pruefteil["abbruch_oder_fail"].astype(int).to_numpy()
quoten = (10, 20, 40, 70, 80)
test_ergebnisse = []

for name, risiko in risiken_test.items():
    ap = average_precision_score(ziel_test, risiko)
    reihenfolge = np.argsort(-risiko, kind="stable")

    for quote in quoten:
        anzahl = int(np.ceil(len(ziel_test) * quote / 100))
        treffer = int(ziel_test[reihenfolge[:anzahl]].sum())

        test_ergebnisse.append({
            "Verfahren": name,
            "Hinweisanteil": f"{quote} %",
            "AP": round(ap, 3),
            "Hinweise": anzahl,
            "Treffer": treffer,
            "Recall": round(treffer / ziel_test.sum(), 3),
            "Trefferquote": round(treffer / anzahl, 3),
            "Übersehen": int(ziel_test.sum() - treffer),
        })

print("Kurseinträge im Prüfteil:", len(pruefteil))
print("Abbruch oder Fail im Prüfteil:", int(ziel_test.sum()))
print("Anteil im Prüfteil:", round(ziel_test.mean(), 3))

display(pd.DataFrame(test_ergebnisse))

Kurseinträge im Prüfteil: 5474
Abbruch oder Fail im Prüfteil: 2437
Anteil im Prüfteil: 0.445


,Verfahren,Hinweisanteil,AP,Hinweise,Treffer,Recall,Trefferquote,Übersehen
0,Referenz: Kurs + Anmeldung,10 %,0.514,548,320,0.131,0.584,2117
1,Referenz: Kurs + Anmeldung,20 %,0.514,1095,599,0.246,0.547,1838
2,Referenz: Kurs + Anmeldung,40 %,0.514,2190,1130,0.464,0.516,1307
3,Referenz: Kurs + Anmeldung,70 %,0.514,3832,1856,0.762,0.484,581
4,Referenz: Kurs + Anmeldung,80 %,0.514,4380,2059,0.845,0.470,378
5,Boosting: Alle Merkmale,10 %,0.709,548,491,0.201,0.896,1946
6,Boosting: Alle Merkmale,20 %,0.709,1095,842,0.346,0.769,1595
7,Boosting: Alle Merkmale,40 %,0.709,2190,1416,0.581,0.647,1021
8,Boosting: Alle Merkmale,70 %,0.709,3832,2029,0.833,0.529,408
9,Boosting: Alle Merkmale,80 %,0.709,4380,2195,0.901,0.501,242


In [10]:
regel_test = (
    pruefteil["assessments_fehlend_bis28"] > 0
).to_numpy()

regel_hinweise = int(regel_test.sum())
regel_treffer = int(ziel_test[regel_test].sum())

print("Einfache Assessment-Regel im Prüfteil")
print("Hinweise:", regel_hinweise)
print("Treffer:", regel_treffer)
print("Hinweisanteil:", round(regel_hinweise / len(pruefteil), 3))
print("Trefferquote:", round(regel_treffer / regel_hinweise, 3))
print("Recall:", round(regel_treffer / ziel_test.sum(), 3))

Einfache Assessment-Regel im Prüfteil
Hinweise: 518
Treffer: 460
Hinweisanteil: 0.095
Trefferquote: 0.888
Recall: 0.189


## Grenzen der Untersuchung – Arbeitsnotiz für Bericht und Präsentation

- **Übertragbarkeit:** OULAD enthält ausgewählte Module der britischen Open
  University aus 2013/2014. Die Ergebnisse belegen keine vergleichbare Güte
  für heutige Kurse oder andere Kontexte der Erwachsenenbildung.
- **Zeitpunkt und Auswahl:** Das Modell betrachtet nur Personen, die am Ende
  von Tag 28 noch angemeldet sind. Frühere Abbrüche kann es nicht erkennen.
  Die Stichtagsdefinition und Ausschlüsse wegen unklarer Datumswerte müssen
  im Bericht nachvollziehbar beschrieben werden.
- **Zwei unterschiedliche Ergebnisse:** „Abbruch oder Fail“ fasst Abmeldung
  und Nichtbestehen zusammen. Die Ursachen und geeigneten Hilfsangebote
  können verschieden sein. Die Erkennungsraten für beide Gruppen müssen
  zusätzlich getrennt betrachtet werden.
- **Begrenzte Merkmale:** Klickzahlen messen erfasste VLE-Nutzung, nicht
  Verständnis oder Motivation. Lernen außerhalb der Plattform fehlt.
  Assessment-Punktzahlen wurden nicht verwendet, weil der Datensatz keinen
  Zeitpunkt enthält, ab dem die jeweilige Bewertung bekannt war.
- **Assessment-Signal:** Ein bis Tag 28 nicht erfülltes fälliges Assessment
  hängt stark mit dem späteren Ergebnis zusammen. Daraus folgt weder die
  Ursache des Ergebnisses noch, dass ein Hinweis den Verlauf ändern würde.
- **Interne Validierung:** Trainiert und geprüft wurde mit unterschiedlichen
  Personen, aber mit Daten derselben historischen Kursumgebung. Eine
  zeitliche Prüfung auf späteren Kursdurchläufen oder eine Prüfung an einer
  anderen Bildungseinrichtung steht aus.
- **Explorative Entscheidungen:** Fragestellung und Merkmale entstanden nach
  Sichtung des gesamten OULAD-Datensatzes. Der zurückgelegte Prüfteil wurde
  nicht für das Modelltraining genutzt, war für die Wahl der Forschungsfrage
  jedoch nicht vollständig unbeachtet. Die Testgüte kann daher optimistisch
  sein.
- **Praktischer Zielkonflikt:** Im Prüfteil erreicht das Modell 90,1 % Recall
  erst bei Hinweisen für 80 % der Personen. Viele Hinweise wären falsch;
  gleichzeitig blieben 242 spätere Fälle unerkannt. Aufwand, mögliche
  Belastung durch Hinweise und geeignete Unterstützungsangebote wurden
  nicht untersucht.
- **Faire Anwendung:** Unterschiede der Güte zwischen Modulen und zwischen
  Personengruppen wurden noch nicht ausreichend geprüft. Ein Hinweis dürfte
  keine automatische negative Entscheidung über eine Person auslösen.
- **Keine Wirkungsstudie:** Die Analyse bewertet Vorhersagen anhand
  historischer Daten. Ob eine Ansprache aufgrund des Modells tatsächlich
  Abbrüche oder Nichtbestehen verhindert, ist damit nicht nachgewiesen.

Quellen: [OULAD-Datenbeschreibung](https://research.stem.open.ac.uk/ouanalyse/open-dataset-more/);
[Kuzilek et al. (2017)](https://doi.org/10.1038/sdata.2017.171).

## Diagnose der Hinweise

Die folgenden Auswertungen beschreiben das bereits festgelegte Modell im
zurückgelegten Prüfteil. Wir ändern aufgrund dieser Ergebnisse weder das Modell
noch die Auswahlgrenzen von 10 % und 20 %.

In [11]:
# final_result wird hier ausschließlich für die Auswertung ergänzt,
# niemals als Eingabe für das Modell verwendet.
info_datei = projektordner / "Daten OULAD" / "studentInfo.csv"
assert info_datei.exists(), f"Datei nicht gefunden: {info_datei}"

info_ergebnis = pd.read_csv(
    info_datei,
    usecols=schluessel + ["final_result"],
)
assert not info_ergebnis.duplicated(schluessel).any()

diagnose = pruefteil[schluessel].copy()
diagnose["zeilen_nr"] = np.arange(len(diagnose))
diagnose["risiko"] = risiken_test["Boosting: Alle Merkmale"]
diagnose["einfache_regel"] = (
    pruefteil["assessments_fehlend_bis28"] > 0
).to_numpy()

diagnose = (
    diagnose.merge(
        info_ergebnis,
        on=schluessel,
        how="left",
        validate="one_to_one",
    )
    .sort_values("zeilen_nr")
    .reset_index(drop=True)
)

assert diagnose["final_result"].notna().all()

diagnose["abbruch_oder_fail"] = diagnose["final_result"].isin(
    ["Withdrawn", "Fail"]
)
assert int(diagnose["abbruch_oder_fail"].sum()) == int(ziel_test.sum())

# Bei 10 % bzw. 20 % erhalten jeweils die höchsten Risikowerte
# des bereits trainierten Modells einen Hinweis.
for anteil in (10, 20):
    anzahl = int(np.ceil(len(diagnose) * anteil / 100))
    positionen = np.argsort(
        -diagnose["risiko"].to_numpy(),
        kind="stable",
    )[:anzahl]

    auswahl = np.zeros(len(diagnose), dtype=bool)
    auswahl[positionen] = True
    diagnose[f"modell_top_{anteil}"] = auswahl

print("Kurseinträge zugeordnet:", len(diagnose))
print(diagnose["final_result"].value_counts())

Kurseinträge zugeordnet: 5474
final_result
Pass           2430
Fail           1409
Withdrawn      1028
Distinction     607
Name: count, dtype: int64


In [12]:
for name, spalte in [
    ("Modell: 10 % Hinweise", "modell_top_10"),
    ("Modell: 20 % Hinweise", "modell_top_20"),
    ("Einfache Assessment-Regel", "einfache_regel"),
]:
    tabelle = diagnose.groupby("final_result").agg(
        kurseintraege=("risiko", "size"),
        mit_hinweis=(spalte, "sum"),
    )

    # Für Fail und Withdrawn entspricht dieser Anteil dem Recall
    # innerhalb der jeweiligen Ergebnisgruppe.
    tabelle["anteil_mit_hinweis_prozent"] = (
        100 * tabelle["mit_hinweis"] / tabelle["kurseintraege"]
    ).round(1)

    print("\n", name)
    display(tabelle)


 Modell: 10 % Hinweise


,kurseintraege,mit_hinweis,anteil_mit_hinweis_prozent
final_result,,,
Distinction,607,6,1.0
Fail,1409,296,21.0
Pass,2430,51,2.1
Withdrawn,1028,195,19.0



 Modell: 20 % Hinweise


,kurseintraege,mit_hinweis,anteil_mit_hinweis_prozent
final_result,,,
Distinction,607,33,5.4
Fail,1409,489,34.7
Pass,2430,220,9.1
Withdrawn,1028,353,34.3



 Einfache Assessment-Regel


,kurseintraege,mit_hinweis,anteil_mit_hinweis_prozent
final_result,,,
Distinction,607,6,1.0
Fail,1409,264,18.7
Pass,2430,52,2.1
Withdrawn,1028,196,19.1


In [13]:
regel = diagnose["einfache_regel"].to_numpy()
modell = diagnose["modell_top_10"].to_numpy()

# So sehen wir, ob das Modell vor allem dieselben Personen markiert
# oder zusätzliche Fälle entdeckt.
diagnose["vergleich_top_10"] = np.select(
    [
        regel & modell,
        regel & ~modell,
        ~regel & modell,
    ],
    [
        "Beide",
        "Nur einfache Regel",
        "Nur Modell",
    ],
    default="Keiner",
)

ueberschneidung = diagnose.groupby("vergleich_top_10").agg(
    kurseintraege=("risiko", "size"),
    abbruch_oder_fail=("abbruch_oder_fail", "sum"),
)

ueberschneidung["anteil_prozent"] = (
    100
    * ueberschneidung["abbruch_oder_fail"]
    / ueberschneidung["kurseintraege"]
).round(1)

display(
    ueberschneidung.reindex(
        ["Beide", "Nur einfache Regel", "Nur Modell", "Keiner"]
    )
)

,kurseintraege,abbruch_oder_fail,anteil_prozent
vergleich_top_10,,,
Beide,467,427,91.4
Nur einfache Regel,51,33,64.7
Nur Modell,81,64,79.0
Keiner,4875,1913,39.2


In [14]:
# Kennzahlen je Modul für die bereits festgelegte globale
# Auswahl der 10 % höchsten Risikowerte.
kursdiagnose = diagnose.assign(
    treffer_top_10=(
        diagnose["modell_top_10"]
        & diagnose["abbruch_oder_fail"]
    )
)

kursvergleich = kursdiagnose.groupby("code_module").agg(
    kurseintraege=("risiko", "size"),
    abbruch_oder_fail=("abbruch_oder_fail", "sum"),
    hinweise_top_10=("modell_top_10", "sum"),
    treffer_top_10=("treffer_top_10", "sum"),
)

kursvergleich["zielanteil_prozent"] = (
    100
    * kursvergleich["abbruch_oder_fail"]
    / kursvergleich["kurseintraege"]
).round(1)

kursvergleich["hinweisanteil_prozent"] = (
    100
    * kursvergleich["hinweise_top_10"]
    / kursvergleich["kurseintraege"]
).round(1)

kursvergleich["trefferquote_prozent"] = (
    100
    * kursvergleich["treffer_top_10"]
    / kursvergleich["hinweise_top_10"].replace(0, np.nan)
).round(1)

kursvergleich["recall_prozent"] = (
    100
    * kursvergleich["treffer_top_10"]
    / kursvergleich["abbruch_oder_fail"]
).round(1)

display(kursvergleich)

,kurseintraege,abbruch_oder_fail,hinweise_top_10,treffer_top_10,zielanteil_prozent,hinweisanteil_prozent,trefferquote_prozent,recall_prozent
code_module,,,,,,,,
AAA,134,35,6,5,26.1,4.5,83.3,14.3
BBB,1284,522,134,121,40.7,10.4,90.3,23.2
CCC,669,352,71,65,52.6,10.6,91.5,18.5
DDD,1052,559,169,151,53.1,16.1,89.3,27.0
EEE,527,198,37,33,37.6,7.0,89.2,16.7
FFF,1330,590,131,116,44.4,9.8,88.5,19.7
GGG,478,181,0,0,37.9,0.0,NaN,0.0


In [15]:
# Jede Validierungsgruppe wird nach Modul UND Präsentation aufgeteilt.
# Innerhalb jeder Kurspräsentation wählen wir die höchsten Risikowerte.
auswahl_je_kurs = []

for fold_nummer, (_, valid_idx) in enumerate(falten, start=1):
    teil = entwicklung.iloc[valid_idx].copy()
    teil["risiko_cv"] = risiken_cv[valid_idx]

    for (modul, praesentation), kurs in teil.groupby(
        ["code_module", "code_presentation"]
    ):
        wahr = kurs["abbruch_oder_fail"].astype(int).to_numpy()
        reihenfolge = np.argsort(
            -kurs["risiko_cv"].to_numpy(),
            kind="stable",
        )

        for anteil in (10, 20):
            anzahl = int(np.ceil(len(kurs) * anteil / 100))
            treffer = int(wahr[reihenfolge[:anzahl]].sum())

            auswahl_je_kurs.append({
                "Fold": fold_nummer,
                "code_module": modul,
                "code_presentation": praesentation,
                "Hinweisanteil": anteil,
                "Kurseinträge": len(kurs),
                "Fälle": int(wahr.sum()),
                "Hinweise": anzahl,
                "Treffer": treffer,
            })

auswahl_je_kurs = pd.DataFrame(auswahl_je_kurs)

# Gesamtleistung dieser kursweisen Vergabe.
gesamt_je_kurs = auswahl_je_kurs.groupby("Hinweisanteil").agg(
    kurseintraege=("Kurseinträge", "sum"),
    faelle=("Fälle", "sum"),
    hinweise=("Hinweise", "sum"),
    treffer=("Treffer", "sum"),
)
gesamt_je_kurs["tatsaechlicher_hinweisanteil"] = (
    gesamt_je_kurs["hinweise"] / gesamt_je_kurs["kurseintraege"]
).round(3)
gesamt_je_kurs["trefferquote"] = (
    gesamt_je_kurs["treffer"] / gesamt_je_kurs["hinweise"]
).round(3)
gesamt_je_kurs["recall"] = (
    gesamt_je_kurs["treffer"] / gesamt_je_kurs["faelle"]
).round(3)

display(gesamt_je_kurs)

# Kontrollblick: Erhält nun jedes Modul Hinweise?
module_bei_10 = (
    auswahl_je_kurs.query("Hinweisanteil == 10")
    .groupby("code_module")
    .agg(
        kurseintraege=("Kurseinträge", "sum"),
        faelle=("Fälle", "sum"),
        hinweise=("Hinweise", "sum"),
        treffer=("Treffer", "sum"),
    )
)
module_bei_10["trefferquote"] = (
    module_bei_10["treffer"] / module_bei_10["hinweise"]
).round(3)

display(module_bei_10)

,kurseintraege,faelle,hinweise,treffer,tatsaechlicher_hinweisanteil,trefferquote,recall
Hinweisanteil,,,,,,,
10,21948,9607,2248,1900,0.102,0.845,0.198
20,21948,9607,4434,3277,0.202,0.739,0.341


,kurseintraege,faelle,hinweise,treffer,trefferquote
code_module,,,,,
AAA,583,151,64,38,0.594
BBB,5130,2141,521,429,0.823
CCC,2819,1459,286,256,0.895
DDD,4206,2092,430,399,0.928
EEE,2020,699,210,173,0.824
FFF,5234,2326,535,473,0.884
GGG,1956,739,202,132,0.653


## Kursweise Hinweise im zurückgelegten Prüfteil

Für die geplante Ansicht wählen wir die höchsten Modellwerte innerhalb jeder
Kurspräsentation aus. Zum Vergleich berechnen wir auch die bisherige Auswahl
über alle Kurspräsentationen hinweg.

Diese Prüfung ist explorativ: Die Idee zur kursweisen Auswahl entstand nach
der Sichtung der globalen Ergebnisse im Prüfteil. Ihre Kennzahlen sind daher
keine neue unabhängige Testbewertung.

In [16]:
# Prüfdaten und bereits berechnete Modellwerte zusammenführen.
auswertung = pruefteil[
    ["code_module", "code_presentation", "abbruch_oder_fail"]
].copy()

auswertung["risiko"] = risiken_test["Boosting: Alle Merkmale"]
wahr = auswertung["abbruch_oder_fail"].astype(bool).to_numpy()

assert len(auswertung) == len(pruefteil)
assert np.isfinite(auswertung["risiko"]).all()

ergebniszeilen = []

for quote in (10, 20):
    # Bisherige Variante: höchste Werte über sämtliche Kurse hinweg.
    anzahl_global = int(np.ceil(len(auswertung) * quote / 100))
    globale_positionen = np.argsort(
        -auswertung["risiko"].to_numpy(),
        kind="stable",
    )[:anzahl_global]

    globaler_hinweis = np.zeros(len(auswertung), dtype=bool)
    globaler_hinweis[globale_positionen] = True

    # Geplante Variante: innerhalb jeder Kurspräsentation auswählen.
    kursweiser_hinweis = np.zeros(len(auswertung), dtype=bool)

    for _, kurs in auswertung.groupby(
        ["code_module", "code_presentation"]
    ):
        anzahl_im_kurs = int(np.ceil(len(kurs) * quote / 100))

        ausgewaehlte_zeilen = (
            kurs.sort_values("risiko", ascending=False, kind="stable")
            .head(anzahl_im_kurs)
            .index
        )
        kursweiser_hinweis[ausgewaehlte_zeilen] = True

    # Beide Varianten mit denselben tatsächlichen Ergebnissen bewerten.
    for verfahren, hinweis in [
        ("Global", globaler_hinweis),
        ("Je Kurspräsentation", kursweiser_hinweis),
    ]:
        hinweise = int(hinweis.sum())
        treffer = int(wahr[hinweis].sum())

        ergebniszeilen.append({
            "Auswahl": verfahren,
            "Geplante Quote": f"{quote} %",
            "Hinweise": hinweise,
            "Tatsächlicher Hinweisanteil": round(
                100 * hinweise / len(auswertung), 1
            ),
            "Treffer": treffer,
            "Trefferquote": round(100 * treffer / hinweise, 1),
            "Recall": round(100 * treffer / wahr.sum(), 1),
        })

    # Für die 10-%-Variante zusätzlich prüfen, wie viele Hinweise
    # jedes Modul bei kursweiser Auswahl erhält.
    if quote == 10:
        module = auswertung.assign(hinweis=kursweiser_hinweis)
        modulhinweise = (
            module.groupby("code_module")["hinweis"]
            .sum()
            .rename("Hinweise bei kursweiser Auswahl")
        )

display(pd.DataFrame(ergebniszeilen))
display(modulhinweise)

,Auswahl,Geplante Quote,Hinweise,Tatsächlicher Hinweisanteil,Treffer,Trefferquote,Recall
0,Global,10 %,548,10.0,491,89.6,20.1
1,Je Kurspräsentation,10 %,558,10.2,474,84.9,19.5
2,Global,20 %,1095,20.0,842,76.9,34.6
3,Je Kurspräsentation,20 %,1102,20.1,803,72.9,33.0


code_module
AAA     14
BBB    131
CCC     68
DDD    107
EEE     54
FFF    135
GGG     49
Name: Hinweise bei kursweiser Auswahl, dtype: int64

### Einordnung der kursweisen Auswahl

Bei 10 % geplanter Hinweisquote erzielt die globale Auswahl im Prüfteil
89,6 % Trefferquote und 20,1 % Recall (548 Hinweise). Die Auswahl innerhalb
jeder Kurspräsentation erzielt 84,9 % Trefferquote und 19,5 % Recall
(558 Hinweise). Bei 20 % geplanter Quote lauten die Werte 76,9 % und
34,6 % (global) gegenüber 72,9 % und 33,0 % (kursweise).

Die kursweise Auswahl verteilt Hinweise auf alle Module; GGG erhält
beispielsweise 49 Hinweise statt zuvor keinen. Sie verbessert damit die
Abdeckung der Kurse, aber nicht die gemessene Vorhersagegüte. Für den
geplanten Prototyp verwenden wir sie als nachvollziehbare Auswahlregel
innerhalb eines gewählten Kursdurchlaufs.

Diese Regel entstand nach Sichtung des zurückgelegten Prüfteils.
Die kursweisen Ergebnisse daraus sind deshalb explorativ und keine
unabhängige Bestätigung ihrer Güte.

In [17]:
# Kennung plus alle Eingabemerkmale des ausgewählten Modells.
demo_spalten = schluessel + [
    spalte for spalte in varianten["Alle Merkmale"]
    if spalte not in schluessel
]

# Nur Merkmale exportieren, die beim Hochladen verfügbar sein sollen.
demo = pruefteil[demo_spalten].copy()

assert "abbruch_oder_fail" not in demo.columns
assert "final_result" not in demo.columns
assert not demo.duplicated(schluessel).any()

demo_ordner = projektordner / "data" / "demo"
demo_ordner.mkdir(parents=True, exist_ok=True)

demo_datei = demo_ordner / "oulad_tag28_demo.csv"
demo.to_csv(demo_datei, index=False)

print("Gespeichert:", demo_datei)
print("Kurseinträge:", len(demo))
print("Module:", demo["code_module"].nunique())
print(
    "Kurspräsentationen:",
    demo[["code_module", "code_presentation"]]
    .drop_duplicates()
    .shape[0]
)
display(demo.groupby("code_module").size().rename("Kurseinträge"))

Gespeichert: c:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\demo\oulad_tag28_demo.csv
Kurseinträge: 5474
Module: 7
Kurspräsentationen: 22


code_module
AAA     134
BBB    1284
CCC     669
DDD    1052
EEE     527
FFF    1330
GGG     478
Name: Kurseinträge, dtype: int64

# Projektkonzept – Frühzeitige Priorisierung von Unterstützung

## Fragestellung
Welche am Ende von Tag 28 noch angemeldeten Lernenden sollten bei
begrenzter Beratungskapazität zuerst ein Unterstützungsangebot erhalten?

## Datengrundlage und Ziel
Wir verwenden OULAD. Betrachtet werden Kurseinträge, die am Ende von
Tag 28 noch angemeldet sind. Das spätere Ergebnis ist „Abbruch oder Fail“.
Als Merkmale dienen ausschließlich bis Tag 28 verfügbare Kurs- und
Anmeldedaten, VLE-Aktivität und Informationen zu Assessments.

## Geplanter Einsatz
Eine Kursverantwortliche betrachtet eine konkrete Kurspräsentation.
Das System ordnet die Lernenden nach geschätztem Risiko und zeigt
beispielhaft die obersten 10 % oder 20 % für eine unterstützende Ansprache.
Die Hinweise sind keine automatische Entscheidung über Teilnehmende.

## Bewertung
Wir vergleichen das Modell mit einer einfachen Assessment-Regel und
einem Modell aus Kurs- und Anmeldedaten. Wichtige Kennzahlen sind die
Trefferquote und der Recall bei festem Hinweisbudget. Personen werden
zwischen Training und Validierung getrennt.

## Aktueller Stand und Grenzen
Ein Datensatz mit 27.422 am Tag 28 noch angemeldeten Kurseinträgen ist
erstellt. Die ersten Ergebnisse zeigen eine brauchbare Priorisierung,
aber erhebliche Unterschiede zwischen Modulen. Die kursweise Vergabe
wurde bislang nur explorativ innerhalb der Entwicklungsdaten bewertet.
Übertragbarkeit und Wirkung einer Ansprache sind nicht nachgewiesen.

## Ergebnis des Portfolio-Projekts
Ein nachvollziehbarer Analyseworkflow, ein begründeter Modellvergleich,
eine einfache Ansicht der priorisierten Fälle und eine kritische
Einordnung der praktischen Grenzen.